# Hybrid Search & Reranking in FFAI's RAG Module

This notebook demonstrates the search subsystem in FFAI's RAG module —
a **pure Python** layer that does **not** require ChromaDB, API keys, or
network access.

We cover:

1. BM25 keyword search
2. Mocked vector search
3. Hybrid search with Reciprocal Rank Fusion (RRF)
4. Standalone RRF for multi-list merging
5. Reranking (Noop / Diversity / CrossEncoder / Factory)
6. Query expansion
7. End-to-end search pipeline
8. Summary table

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

print(f'Project root: {_project_root}')

In [ ]:
from ffai.rag.embed import Embeddings
from ffai.rag.indexing import BM25Index
from ffai.rag.search import (
    CrossEncoderReranker,
    DiversityReranker,
    HybridSearch,
    NoopReranker,
    QueryExpander,
    fuse_search_results,
    get_reranker,
    reciprocal_rank_fusion,
)

print('All imports successful')

<div class="page-break"></div>

---

## Section 2: BM25 Keyword Search

BM25 (Okapi BM25) is a bag-of-words ranking function that scores
documents based on term frequency, inverse document frequency, and
document length normalization. It excels at exact keyword matching.

We index 10 documents covering different programming topics
(Python async, Rust ownership, Go goroutines, JavaScript promises, etc.).

In [ ]:
# Build a BM25 index with 10 programming-topic documents
bm25 = BM25Index()

docs = [
    ("python_async", "Python asyncio provides asynchronous programming with async/await syntax. "
     "Coroutines and event loops enable concurrent I/O operations without threads."),
    ("rust_ownership", "Rust ownership model ensures memory safety without garbage collection. "
     "The borrow checker enforces strict rules about variable lifetimes and references."),
    ("go_goroutines", "Go goroutines are lightweight concurrency primitives managed by the Go runtime. "
     "Channels provide safe communication between goroutines for concurrent programming."),
    ("js_promises", "JavaScript promises handle asynchronous operations in single-threaded environments. "
     "Async/await syntax simplifies promise chains for concurrent task management."),
    ("java_threads", "Java threads enable concurrent programming with shared memory. "
     "Synchronized blocks and locks prevent race conditions in multi-threaded code."),
    ("cpp_memory", "C++ smart pointers provide memory safety through RAII patterns. "
     "Unique pointers enforce ownership while shared pointers use reference counting."),
    ("rust_concurrency", "Rust concurrency combines ownership with threads for safe parallel execution. "
     "The type system prevents data races at compile time through Send and Sync traits."),
    ("elixir_actor", "Elixir uses the actor model for concurrent programming via lightweight processes. "
     "Message passing between processes eliminates shared-state concurrency bugs."),
    ("python_error", "Python exception handling uses try/except/finally blocks for error management. "
     "Custom exception classes enable structured error propagation in async code."),
    ("kotlin_coroutines", "Kotlin coroutines simplify asynchronous programming on the JVM. "
     "Structured concurrency ensures all launched coroutines complete predictably."),
]

for doc_id, content in docs:
    bm25.add_document(doc_id, content)

print(f'Indexed {bm25.count()} documents')
print(f'Index stats: {bm25.get_stats()}')

The index now holds 10 documents with 126 unique terms.
The average document length is ~19.5 tokens. Default BM25 parameters are
`k1=1.5` (term frequency saturation) and `b=0.75` (length normalization).

In [ ]:
# Search for "concurrent programming" — should surface concurrency-focused docs
results = bm25.search('concurrent programming', n_results=5)
print('Query: "concurrent programming"')
print('-' * 70)
for r in results:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
    print(f"    {r['content'][:90]}...")
print()

Top result is `python_async` (score 1.44) because it mentions both
"concurrent" and "programming". Documents that explicitly contain
"concurrent programming" as a phrase score highest.

In [ ]:
# Search for "memory safety" — should surface Rust/C++ docs
results = bm25.search('memory safety', n_results=5)
print('Query: "memory safety"')
print('-' * 70)
for r in results:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
    print(f"    {r['content'][:90]}...")
print()

`cpp_memory` (2.72) and `rust_ownership` (2.60) dominate because they
both directly discuss "memory safety". Other documents score much lower
since they lack these specific terms.

In [ ]:
# Multi-word queries: BM25 tokenizes and scores per term
results_async = bm25.search('async error handling', n_results=5)
print('Query: "async error handling"')
print('-' * 70)
for r in results_async:
    print(f"  {r['id']:25s}  score={r['score']:.4f}")
print()
print('Notice: docs containing multiple query terms score higher')
print('(python_async matches "async", python_error matches "error" & "handling")')

`python_error` scores 5.81 — far above the rest — because it contains
both "error" and "handling". BM25 sums per-term scores, so documents
matching more query terms receive disproportionately higher scores.

<div class="page-break"></div>

---

## Section 3: Vector Search (Mocked)

Vector search compares semantic meaning via embedding similarity.
Here we mock the embedding function with deterministic fake vectors
so no model download or API key is needed.

In [ ]:
import hashlib
import math


def mock_embed(text: str, dim: int = 64) -> list[float]:
    """Deterministic fake embedding from token hashes."""
    tokens = text.lower().split()
    vec = [0.0] * dim
    for token in tokens:
        h = int(hashlib.md5(token.encode()).hexdigest(), 16)
        for j in range(dim):
            angle = ((h + j * 17) % 10000) / 10000.0 * 2 * math.pi
            vec[j] += math.cos(angle)
    norm = math.sqrt(sum(x * x for x in vec)) or 1.0
    return [x / norm for x in vec]


# Pre-compute embeddings for all documents
doc_contents = dict(docs)
doc_embeddings = {doc_id: mock_embed(content) for doc_id, content in docs}

print(f'Pre-computed {len(doc_embeddings)} mock embeddings, dimension={len(next(iter(doc_embeddings.values())))}')

In [ ]:
def vector_search_fn(query: str, n_results: int) -> list[dict]:
    """Mock vector search using cosine similarity against pre-computed embeddings."""
    query_emb = mock_embed(query)
    scored = []
    for doc_id, doc_emb in doc_embeddings.items():
        sim = Embeddings.cosine_similarity(query_emb, doc_emb)
        scored.append({
            'id': doc_id,
            'content': doc_contents[doc_id],
            'score': sim,
            'metadata': {},
        })
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:n_results]


vec_results = vector_search_fn('concurrent programming', 5)
print('Vector search: "concurrent programming"')
print('-' * 70)
for r in vec_results:
    print(f"  {r['id']:25s}  score={r['score']:.6f}")
    print(f"    {r['content'][:90]}...")

Vector search ranks `rust_concurrency` (0.9995) first, followed by
`python_async` (0.9982). Note these rankings differ from BM25 — vector
search captures token-level overlap patterns rather than exact term matches.

<div class="page-break"></div>

---

## Section 4: Hybrid Search with Reciprocal Rank Fusion

Hybrid search combines BM25 keyword matching and vector semantic
similarity via **Reciprocal Rank Fusion** (RRF). The `alpha` parameter
controls the weight: `alpha` favors vector search, `1-alpha` favors BM25.

RRF score = `alpha/(k + vector_rank) + (1-alpha)/(k + bm25_rank)`.

In [ ]:
# Wrap BM25 as a search function for HybridSearch
def bm25_search_fn(query: str, n_results: int) -> list[dict]:
    return bm25.search(query, n_results=n_results)


# Create HybridSearch with default alpha=0.6 (slightly favors vector)
hybrid = HybridSearch(
    vector_search_fn=vector_search_fn,
    bm25_search_fn=bm25_search_fn,
    alpha=0.6,
    rrf_k=60,
)

results = hybrid.search('error handling in async code', n_results=5)
print('Hybrid search: "error handling in async code" (alpha=0.6)')
print('-' * 90)
print(f"  {'ID':25s}  {'vec_score':>10s}  {'bm25_score':>10s}  {'rrf_score':>10s}")
print('-' * 90)
for r in results:
    vs = r.get('vector_score', 0)
    bs = r.get('bm25_score', 0)
    rrf = r.get('rrf_score', 0)
    print(f"  {r['id']:25s}  {vs:10.6f}  {bs:10.4f}  {rrf:10.6f}")

`python_error` ranks first (rrf=0.0162) because it scores well in both
vector search (0.999) and BM25 (8.35). The hybrid approach surfaces it
higher than either individual method might alone.

In [ ]:
# Compare alpha values: vector-heavy vs BM25-heavy
print('Effect of alpha on hybrid ranking for: "error handling in async code"')
print('=' * 90)
for alpha_val in [0.9, 0.6, 0.3]:
    hybrid.set_alpha(alpha_val)
    results = hybrid.search('error handling in async code', n_results=3)
    print(f'\nalpha={alpha_val:.1f} (vector weight={alpha_val:.0%}, bm25 weight={1-alpha_val:.0%}):')
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['id']:25s}  rrf_score={r['rrf_score']:.6f}")

hybrid.set_alpha(0.6)  # reset

With `alpha=0.9` (vector-heavy), `python_error` and `python_async` lead.
With `alpha=0.3` (BM25-heavy), `java_threads` rises to rank 1 because
it has a high BM25 score (2.66) from matching "error" and "handling".
Tuning `alpha` lets you balance semantic vs. keyword relevance.

<div class="page-break"></div>

---

## Section 5: Reciprocal Rank Fusion (Standalone)

The standalone `reciprocal_rank_fusion()` function merges **arbitrary** result
lists — not just vector+BM25. RRF scores each document as
`sum(weight / (k + rank))` across all lists, so documents appearing near
the top of multiple lists get the highest combined score.

In [ ]:
# Three result lists from three different queries about the same topic
list_a = [
    {'id': 'python_async', 'content': 'Python asyncio ...'},
    {'id': 'rust_concurrency', 'content': 'Rust concurrency ...'},
    {'id': 'go_goroutines', 'content': 'Go goroutines ...'},
]

list_b = [
    {'id': 'go_goroutines', 'content': 'Go goroutines ...'},
    {'id': 'elixir_actor', 'content': 'Elixir actor model ...'},
    {'id': 'python_async', 'content': 'Python asyncio ...'},
]

list_c = [
    {'id': 'kotlin_coroutines', 'content': 'Kotlin coroutines ...'},
    {'id': 'python_async', 'content': 'Python asyncio ...'},
    {'id': 'rust_concurrency', 'content': 'Rust concurrency ...'},
]

fused = reciprocal_rank_fusion([list_a, list_b, list_c], k=60)
print('Fused results from 3 lists (k=60):')
print('-' * 50)
for r in fused:
    print(f"  {r['id']:25s}  rrf_score={r['rrf_score']:.6f}")

print()
print('"python_async" appears in all 3 lists at various ranks → highest RRF score')
print('"kotlin_coroutines" appears in only 1 list → lowest RRF score')

`python_async` wins (0.0161) because it appears in all 3 lists at
ranks 1, 3, and 2. Documents appearing in multiple lists accumulate
RRF points from each, while single-list appearances (like
`kotlin_coroutines` at 0.0055) score much lower.

<div class="page-break"></div>

---

## Section 6: Reranking

Rerankers take an initial result set and reorder it using a second-pass
scoring strategy. FFAI ships three rerankers:

- **NoopReranker**: pass-through (no changes)
- **DiversityReranker**: promotes varied results via MMR-style selection
- **CrossEncoderReranker**: neural cross-encoder scoring

### 6a. NoopReranker

In [ ]:
noop = NoopReranker()
sample = [
    {'id': 'a', 'content': 'doc a', 'score': 0.9},
    {'id': 'b', 'content': 'doc b', 'score': 0.7},
    {'id': 'c', 'content': 'doc c', 'score': 0.5},
]
reranked = noop.rerank('test query', sample)
print('NoopReranker — pass-through (no reordering):')
for r in reranked:
    print(f"  {r['id']}  score={r['score']}")
print(f'Results unchanged: {reranked == sample}')

The `NoopReranker` returns results in the same order and content. It is
useful as a placeholder when reranking is disabled but the interface is
still required.

### 6b. DiversityReranker

In [ ]:
# Create results with overlapping vocabulary to show diversity effect
overlap_results = [
    {'id': 'r1', 'content': 'Python async await coroutine concurrent task', 'score': 0.95},
    {'id': 'r2', 'content': 'Python async await coroutine event loop', 'score': 0.90},
    {'id': 'r3', 'content': 'Python async await future callback', 'score': 0.85},
    {'id': 'r4', 'content': 'Rust ownership borrow checker lifetime', 'score': 0.70},
    {'id': 'r5', 'content': 'Go goroutine channel message passing', 'score': 0.60},
]

print('BEFORE DiversityReranker (lambda=0.7):')
for i, r in enumerate(overlap_results, 1):
    print(f"  {i}. {r['id']}  score={r['score']:.2f}  {r['content'][:50]}")

div_reranker = DiversityReranker(lambda_param=0.7)
diverse = div_reranker.rerank('async programming', overlap_results)

print('\nAFTER DiversityReranker (lambda=0.7):')
for i, r in enumerate(diverse, 1):
    print(f"  {i}. {r['id']}  diversity_rank={r['diversity_rank']}  score={r['score']:.2f}  {r['content'][:50]}")

print('\nTop result stays (highest relevance), then diverse docs promoted over similar ones')

Before: r1, r2, r3 (all Python async docs) occupied the top 3 spots.
After: r1 stays first (highest relevance), but r4 (Rust) jumps to rank 2
because it has low word overlap with r1. r2 and r3 (similar to r1)
get pushed down. The `lambda_param` balances relevance vs. diversity
(0.7 = 70% relevance, 30% diversity penalty).

### 6c. CrossEncoderReranker (Mocked)

In [ ]:
import types

# Mock sentence_transformers.CrossEncoder so no download is needed
mock_ce_module = types.ModuleType('sentence_transformers')


class _MockCrossEncoder:
    def __init__(self, model_name, **kw):
        self.model_name = model_name

    def predict(self, pairs):
        import random
        rng = random.Random(42)
        return [rng.uniform(0.3, 0.99) for _ in pairs]


mock_ce_module.CrossEncoder = _MockCrossEncoder
sys.modules['sentence_transformers'] = mock_ce_module

# Now CrossEncoderReranker can load the mock
ce_reranker = CrossEncoderReranker(model_name='mock-model')

ce_input = [
    {'id': 'd1', 'content': 'async await in Python', 'score': 0.8},
    {'id': 'd2', 'content': 'Rust borrow checker rules', 'score': 0.6},
    {'id': 'd3', 'content': 'Go channels and goroutines', 'score': 0.5},
]

ce_results = ce_reranker.rerank('async programming', ce_input)
print('CrossEncoderReranker (mocked) results:')
for r in ce_results:
    print(f"  {r['id']}  rerank_score={r['rerank_score']:.4f}  original_score={r['original_score']}")

The cross-encoder scores each (query, document) pair and reorders by
`rerank_score`. In production you would use a real model like
`cross-encoder/ms-marco-MiniLM-L-6-v2`. Here we mock it with seeded
random scores to demonstrate the API without downloading a model.

### 6d. Factory: `get_reranker()`

In [ ]:
# Factory dispatches by type string
for rtype in ('none', 'diversity', 'cross_encoder'):
    r = get_reranker(rtype)
    print(f"  get_reranker('{rtype}') → {type(r).__name__}")

`get_reranker()` returns the correct class based on a type string:
`"none"` → NoopReranker, `"diversity"` → DiversityReranker,
`"cross_encoder"` → CrossEncoderReranker.

<div class="page-break"></div>

---

## Section 7: Query Expansion

`QueryExpander` uses an LLM to reformulate a query into multiple
variations, improving recall by catching different phrasings.
When no LLM function is configured, it falls back to the original query.

In [ ]:
# Mock LLM that returns numbered query variations
def mock_llm(prompt: str) -> str:
    return (
        "1. What are different ways to authenticate users\n"
        "2. Authentication protocols and mechanisms\n"
        "3. Login methods and session management techniques"
    )


expander = QueryExpander(llm_generate_fn=mock_llm, n_variations=3)
expanded = expander.expand('authentication methods')
print('Original query: "authentication methods"')
print(f'Expanded to {len(expanded)} queries:')
for i, q in enumerate(expanded):
    print(f'  {i + 1}. {q}')

The expander produces 4 queries: the original plus 3 LLM-generated
variations. Each variation uses different wording to catch documents
the original query might miss.

In [ ]:
# Without an LLM function, returns original only
no_llm_expander = QueryExpander(n_variations=3)
fallback = no_llm_expander.expand('authentication methods')
print(f'No LLM configured → {len(fallback)} query: "{fallback[0]}"')

In [ ]:
# Use fuse_search_results to merge results from multiple expanded queries
results_q1 = [
    {'id': 'auth_1', 'content': 'OAuth 2.0 flow', 'score': 0.95},
    {'id': 'auth_2', 'content': 'JWT tokens', 'score': 0.80},
]
results_q2 = [
    {'id': 'auth_3', 'content': 'SAML SSO integration', 'score': 0.90},
    {'id': 'auth_1', 'content': 'OAuth 2.0 flow', 'score': 0.85},  # duplicate
]
results_q3 = [
    {'id': 'auth_4', 'content': 'API key management', 'score': 0.75},
    {'id': 'auth_2', 'content': 'JWT tokens', 'score': 0.70},  # duplicate
]

fused = fuse_search_results([results_q1, results_q2, results_q3], n_results=5)
print('Fused results from 3 expanded queries (deduplicated):')
for r in fused:
    print(f"  {r['id']}  score={r['score']:.2f}  {r['content']}")
print(f'\n{len(fused)} unique results (auth_1 and auth_2 duplicates removed)')

`fuse_search_results()` merges results from multiple queries while
deduplicating by `id`. The first occurrence wins, preserving the score
from the highest-priority query. Here 6 results across 3 queries become
4 unique results.

<div class="page-break"></div>

---

## Section 8: End-to-End Search Pipeline

Put it all together: BM25 index → Hybrid search → Reranking.

In [ ]:
# Full pipeline for a single query
query = 'safe concurrent error handling'

# Stage 1: Hybrid search (BM25 + vector via RRF)
hybrid.set_alpha(0.5)
stage1 = hybrid.search(query, n_results=8, mode='hybrid')
print(f'Stage 1 — Hybrid search for "{query}"')
print('-' * 70)
for i, r in enumerate(stage1, 1):
    print(f"  {i}. {r['id']:25s}  rrf={r['rrf_score']:.6f}")

Stage 1 returns 8 results fused from vector and BM25 search.
`rust_concurrency` (0.0161) and `python_error` (0.0160) lead because
they match well across both modalities for this multi-topic query.

In [ ]:
# Stage 2: Diversity reranking to promote varied topics
stage2 = div_reranker.rerank(query, stage1, n_results=5)
print('Stage 2 — After DiversityReranker (lambda=0.7)')
print('-' * 70)
for i, r in enumerate(stage2, 1):
    print(f"  {i}. {r['id']:25s}  diversity_rank={r['diversity_rank']}  rrf={r.get('rrf_score', 0):.6f}")
    print(f"     {r['content'][:80]}...")

After diversity reranking, `cpp_memory` (originally rank 8) jumps to
rank 3 because it covers memory safety — a distinct topic from the
concurrency-focused results above it. The pipeline ensures the user sees
results covering **different aspects** of "safe concurrent error handling".

<div class="page-break"></div>

---

## Section 9: Summary

In [ ]:
import polars as pl

summary = pl.DataFrame([
    {'Component': 'BM25Index', 'Purpose': 'Keyword search via Okapi BM25', 'Key Parameters': 'k1, b, epsilon'},
    {'Component': 'HybridSearch', 'Purpose': 'Combine vector + BM25 via RRF', 'Key Parameters': 'alpha, rrf_k, mode'},
    {'Component': 'reciprocal_rank_fusion', 'Purpose': 'Merge arbitrary result lists', 'Key Parameters': 'k, weights'},
    {'Component': 'NoopReranker', 'Purpose': 'Pass-through (disabled reranking)', 'Key Parameters': '\u2014'},
    {'Component': 'DiversityReranker', 'Purpose': 'Promote diverse results via MMR', 'Key Parameters': 'lambda_param'},
    {'Component': 'CrossEncoderReranker', 'Purpose': 'Cross-encoder neural reranking', 'Key Parameters': 'model_name, max_length'},
    {'Component': 'QueryExpander', 'Purpose': 'Multi-query retrieval via LLM', 'Key Parameters': 'n_variations, include_original'},
    {'Component': 'fuse_search_results', 'Purpose': 'Deduplicate merged results', 'Key Parameters': 'n_results, dedupe_by'},
])

print(summary)